In [ ]:
import os, pandas as pd
from scipy.stats import randint
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

paths = ["./datasets/DDoS.csv","../datasets/DDoS.csv","../../datasets/DDoS.csv","DDoS.csv"]
csv = next((p for p in paths if os.path.exists(p)), None)
if not csv: raise FileNotFoundError("Coloque DDoS.csv em ./datasets/")

try: df = pd.read_csv(csv, encoding="utf-8")
except UnicodeDecodeError: df = pd.read_csv(csv, encoding="latin1")

for c in ["source IP","dest IP","source port","Dest Port"]:
    if c in df.columns: df = df.drop(columns=[c])

y = df["target"].astype(int)
X = df.drop(columns=["target"])

cat = X.select_dtypes(include=["object","category"]).columns
num = X.select_dtypes(include=["number"]).columns

try: ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError: ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

prep = ColumnTransformer([("num", StandardScaler(with_mean=False), num),
                          ("cat", ohe, cat)], remainder="drop")

pipe = Pipeline([("prep", prep), ("clf", RandomForestClassifier(random_state=42))])

cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, scoring="accuracy", cv=cv, n_jobs=-1)
print(f"[KFold-5] acc: {scores.mean():.4f} ± {scores.std():.4f}")

param_dist = {
    "clf__n_estimators": randint(150, 600),
    "clf__max_depth": [None, 6, 10, 16, 24],
    "clf__min_samples_split": randint(2, 12),
    "clf__min_samples_leaf": randint(1, 8),
    "clf__max_features": ["sqrt", "log2", 0.5],
}
rs = RandomizedSearchCV(pipe, param_dist, n_iter=20, scoring="accuracy",
                        cv=cv, n_jobs=-1, random_state=42, verbose=1)
rs.fit(X, y)
print(f"[RandomSearch] best_acc: {rs.best_score_:.4f}\nparams: {rs.best_params_}")


[KFold-5] acc: 1.0000 ± 0.0000
Fitting 5 folds for each of 20 candidates, totalling 100 fits
